# วัดขนาดวัตถุจากภาพอย่างละเอียด
notebook 06 วัดได้ละเอียด 1 พิกเซล  ในงานจริงต้องรู้ว่า **ค่าที่วัดได้เชื่อถือได้แค่ไหน และทำให้ละเอียดกว่า 1 พิกเซลได้อย่างไร**

| ขั้น | เรื่อง | ผลที่ได้ |
|---|---|---|
| 1 | พิกเซลเป็น มม. ได้อย่างไร และผิดได้ตรงไหน | เข้าใจแหล่งความคลาดเคลื่อน |
| 2 | สอบเทียบด้วยวัตถุอ้างอิง (วิธีง่าย) | px/mm 1 ค่า |
| 3 | แก้มุมกล้องเอียงด้วยจุดอ้างอิง 4 จุด (Homography) | วัดถูกแม้กล้องไม่ตั้งฉาก |
| 4 | หาขอบละเอียดกว่าพิกเซล (Sub-pixel edge) | ละเอียด 0.1–0.2 พิกเซล |
| 5 | ทดสอบความทำซ้ำได้ (Repeatability) | ค่าเฉลี่ย ± 3σ ที่รายงานลูกค้าได้ |
| 6 | เช็กลิสต์ก่อนวัดจริง และขีดจำกัด | รู้ว่าเมื่อไรต้องใช้เลนส์ Telecentric |

ภาพทั้งหมดในนี้สร้างขึ้นเพื่อให้**รู้ขนาดจริงแน่นอน** จะได้เทียบว่าวิธีไหนแม่นแค่ไหน

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import cv2
import numpy as np
from vision_camera import show
rng = np.random.default_rng(0)
print('พร้อม')

## 1. พิกเซลเป็นมิลลิเมตรได้อย่างไร
กล้องเห็นเป็นพิกเซล  ขนาดจริง = พิกเซล ÷ (px/mm)  ค่า px/mm ขึ้นกับ**ระยะกล้องถึงวัตถุ**และเลนส์  แหล่งความคลาดเคลื่อนหลัก 5 อย่าง

| แหล่ง | ขนาดความผิดพลาดโดยทั่วไป | แก้ด้วย |
|---|---|---|
| ความละเอียดพิกเซล | ±0.5 px | กล้องละเอียดขึ้น / ซูมเข้า / Sub-pixel (ขั้น 4) |
| วัตถุสูงไม่เท่าวัตถุสอบเทียบ (Parallax) | 1–5% | วางระนาบเดียวกัน / เลนส์ Telecentric |
| กล้องเอียง (Perspective) | 1–10% ต่างกันคนละฝั่งภาพ | ตั้งฉาก / Homography (ขั้น 3) |
| เลนส์บิดเบี้ยว (Distortion) | 0.5–3% ที่ขอบภาพ | cv2.calibrateCamera ด้วยลายหมากรุก |
| ขอบไม่คม (เบลอ แสง เงา) | 1–3 px | Backlight, โฟกัสคม, Threshold ที่ 50% |

**กฎง่ายๆ:** ความแม่นที่ทำได้จริงราว **1/10 ของขนาดพิกเซล** เมื่อทำทุกอย่างถูก และไม่ดีกว่า **1 พิกเซล** ถ้าไม่ทำอะไรเลย

## 2. สอบเทียบด้วยวัตถุอ้างอิง (วิธีง่าย ใช้ได้เมื่อกล้องตั้งฉาก)
สร้างภาพ: เหรียญเส้นผ่านศูนย์กลาง **20.0 มม.** และชิ้นงาน **25.50 × 16.00 มม.** ถ่ายที่ **11.7 px/mm** (เรารู้คำตอบ)  ภาพมีขอบเบลอและจุดรบกวนแบบกล้องจริง

In [ ]:
PPM_TRUE = 11.7                       # px/mm ของภาพจำลอง (ตั้งใจให้ขอบไม่ลงพอดีพิกเซล เหมือนของจริง)
PART_W, PART_H = 25.5, 16.0           # มม. (คำตอบจริง)

def make_scene(ppm=PPM_TRUE, noise=12.0, blur=3.0, S=4):
    # วาดที่ความละเอียด S เท่า แล้วย่อลง เพื่อให้ขอบอยู่ระหว่างพิกเซลได้ (anti-aliasing) เหมือนภาพจากกล้องจริง
    big = np.full((480 * S, 640 * S, 3), 235, np.uint8)
    cv2.circle(big, (130 * S, 240 * S), int(round(10.0 * ppm * S)), (60, 60, 60), -1)         # เหรียญ 20 มม.
    x0, y0 = 430 * S, 240 * S
    cv2.rectangle(big, (int(round(x0 - PART_W * ppm * S / 2)), int(round(y0 - PART_H * ppm * S / 2))),
                  (int(round(x0 + PART_W * ppm * S / 2)), int(round(y0 + PART_H * ppm * S / 2))), (70, 70, 70), -1)
    img = cv2.resize(big, (640, 480), interpolation=cv2.INTER_AREA)
    if blur:
        img = cv2.GaussianBlur(img, (0, 0), blur)
    return np.clip(img + rng.normal(0, noise, img.shape), 0, 255).astype(np.uint8)

def measure_simple(img, coin_mm=20.0):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cs = sorted([c for c in cs if cv2.contourArea(c) > 500], key=lambda c: cv2.boundingRect(c)[0])
    (cx, cy), r = cv2.minEnclosingCircle(cs[0])
    ppm = 2 * r / coin_mm
    w, h = cv2.minAreaRect(cs[1])[1]
    W, H = sorted([w / ppm, h / ppm], reverse=True)
    return ppm, W, H

img = make_scene()
ppm, W, H = measure_simple(img)
print(f'สอบเทียบได้ {ppm:.3f} px/mm (จริง {PPM_TRUE})   ชิ้นงาน {W:.2f} x {H:.2f} มม. (จริง {PART_W} x {PART_H})')
print(f'ผิด {abs(W - PART_W):.2f} และ {abs(H - PART_H):.2f} มม.  = ราว {abs(W - PART_W) * ppm:.1f} พิกเซล')
show(img, titles=['ภาพสอบเทียบ: เหรียญซ้าย ชิ้นงานขวา'], size=5)

ผิดราว 1 พิกเซล มาจาก Threshold ที่ตัดขอบเบลอ และ `minAreaRect` ที่ล้อมรอบพิกเซลขอบ  ถ้าชิ้นงานใหญ่ขึ้น ความผิดพลาดเป็นมิลลิเมตรเท่าเดิม แต่เป็น % น้อยลง

## 3. กล้องเอียง: แก้ด้วยจุดอ้างอิง 4 จุด (Homography)
ในโรงงาน กล้องมักตั้งฉากไม่สมบูรณ์  ภาพจะบิดเป็นสี่เหลี่ยมคางหมู ฝั่งใกล้ดูใหญ่ ฝั่งไกลดูเล็ก
วิธีแก้: ติด**จุดอ้างอิง 4 จุด**บนจิ๊กที่รู้ระยะจริง (เช่น สี่เหลี่ยม 100 × 60 มม.)  แล้วแปลงภาพให้เป็นมุมมองตั้งฉากก่อนวัด

In [ ]:
FID_MM = np.float32([[0, 0], [100, 0], [100, 60], [0, 60]])   # ตำแหน่งจุดอ้างอิงจริง (มม.)

def make_tilted(tilt=0.25):
    # ฉากตั้งฉาก: จิ๊ก 100x60 มม. ที่ 5 px/mm มีจุดอ้างอิง 4 มุม และชิ้นงานตรงกลาง
    ppm = 5.0
    flat = np.full((400, 600, 3), 235, np.uint8)
    off = np.array([50, 50])
    for x, y in FID_MM:
        cv2.circle(flat, (int(off[0] + x * ppm), int(off[1] + y * ppm)), 6, (30, 30, 30), -1)
    cx, cy = off + np.array([50, 30]) * ppm
    cv2.rectangle(flat, (int(cx - PART_W * ppm / 2), int(cy - PART_H * ppm / 2)),
                  (int(cx + PART_W * ppm / 2), int(cy + PART_H * ppm / 2)), (70, 70, 70), -1)
    # จำลองกล้องเอียง: บีบด้านบนให้แคบลง
    h, w = flat.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([[w * tilt, 0], [w * (1 - tilt), 0], [w, h], [0, h]])
    M = cv2.getPerspectiveTransform(src, dst)
    tilted = cv2.warpPerspective(flat, M, (w, h), borderValue=(235, 235, 235))
    tilted = cv2.GaussianBlur(tilted, (0, 0), 1.0)
    return np.clip(tilted + rng.normal(0, 3, tilted.shape), 0, 255).astype(np.uint8)

def find_fiducials(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    dots = [c for c in cs if 20 < cv2.contourArea(c) < 400]
    pts = np.float32([cv2.minEnclosingCircle(c)[0] for c in dots])
    # เรียงเป็น ซ้ายบน ขวาบน ขวาล่าง ซ้ายล่าง
    s = pts.sum(1); d = pts[:, 0] - pts[:, 1]
    return np.float32([pts[s.argmin()], pts[d.argmax()], pts[s.argmax()], pts[d.argmin()]])

def rectify(img, ppm_out=8.0):
    pts = find_fiducials(img)
    dst = FID_MM * ppm_out + 40
    H = cv2.getPerspectiveTransform(pts, dst)
    out = cv2.warpPerspective(img, H, (int(100 * ppm_out + 80), int(60 * ppm_out + 80)), borderValue=(235, 235, 235))
    return out, ppm_out, pts

def measure_part(img, ppm):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cs, key=cv2.contourArea)
    w, h = cv2.minAreaRect(c)[1]
    return sorted([w / ppm, h / ppm], reverse=True)

tilted = make_tilted()
# วัดตรงๆ โดยใช้ระยะจุดอ้างอิงด้านล่าง (ใกล้กล้อง) เป็น px/mm
pts = find_fiducials(tilted)
ppm_naive = np.linalg.norm(pts[2] - pts[3]) / 100.0
W1, H1 = measure_part(tilted, ppm_naive)
fixed, ppm_fix, _ = rectify(tilted)
W2, H2 = measure_part(fixed, ppm_fix)
print(f'ไม่แก้มุม:  {W1:.2f} x {H1:.2f} มม.   ผิด {abs(W1 - PART_W):.2f} / {abs(H1 - PART_H):.2f} มม.')
print(f'แก้ด้วย Homography:  {W2:.2f} x {H2:.2f} มม.   ผิด {abs(W2 - PART_W):.2f} / {abs(H2 - PART_H):.2f} มม.')
vis = tilted.copy()
for p in pts:
    cv2.circle(vis, (int(p[0]), int(p[1])), 10, (0, 0, 255), 2)
show(vis, fixed, titles=['กล้องเอียง (วงแดง = จุดอ้างอิงที่หาเจอ)', 'หลังแปลงเป็นมุมตั้งฉาก'], size=5)

**บทเรียน:** กล้องเอียงไม่มาก ก็ทำให้วัดผิดเป็นมิลลิเมตร  จุดอ้างอิง 4 จุดบนจิ๊กแก้ได้เกือบหมด และยัง**ตรวจสอบตัวเองได้**: ถ้าระยะระหว่างจุดอ้างอิงหลังแปลงไม่ตรง 100 × 60 แสดงว่ากล้องหรือจิ๊กขยับ
ข้อจำกัด: ใช้ได้กับวัตถุ**แบน**ที่อยู่ระนาบเดียวกับจุดอ้างอิง  วัตถุสูงต้องใช้เลนส์ Telecentric หรือกล้อง 3 มิติ

## 4. ละเอียดกว่าพิกเซล: Sub-pixel edge
Threshold ตัดขอบที่พิกเซลเต็ม  แต่ขอบจริงอยู่ **ระหว่าง** พิกเซล  ดูจากความสว่างตามแนวขวางขอบ: สว่าง → ค่อยๆ มืด  จุดที่ความสว่างผ่าน**ครึ่งทาง (50%)** คือตำแหน่งขอบที่แท้จริง หาได้เป็นทศนิยม

In [ ]:
def edge_50(profile, bright, dark):
    # หาตำแหน่งที่ความสว่างผ่านครึ่งทางระหว่าง bright กับ dark (ทศนิยม) จากซ้ายไปขวา
    mid = (bright + dark) / 2.0
    p = profile.astype(np.float32)
    for i in range(len(p) - 1):
        if (p[i] - mid) * (p[i + 1] - mid) <= 0 and p[i] != p[i + 1]:
            return i + (p[i] - mid) / (p[i] - p[i + 1])
    return None

def width_subpixel(img, c, n_lines=15):
    # วัดความกว้างชิ้นงานสี่เหลี่ยมตามแนวนอน ด้วยเส้นสแกน n เส้น แล้วเฉลี่ย
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    x, y, w, h = cv2.boundingRect(c)
    bright = float(np.median(gray[y + h // 2, max(0, x - 30):x - 12]))
    dark = float(np.median(gray[y + h // 2, x + 12:x + w - 12]))
    widths = []
    for yy in np.linspace(y + h * 0.2, y + h * 0.8, n_lines).astype(int):
        row = gray[yy, x - 10:x + w + 10]
        left = edge_50(row, bright, dark)
        right = edge_50(row[::-1], bright, dark)
        if left is not None and right is not None:
            widths.append((len(row) - 1 - right) - left)
    return float(np.mean(widths))

def contour_of_part(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return sorted([c for c in cs if cv2.contourArea(c) > 500], key=lambda c: cv2.boundingRect(c)[0])[1]

img = make_scene()
ppm, W_px, _ = measure_simple(img)
c = contour_of_part(img)
w_rect = max(cv2.minAreaRect(c)[1])
w_sub = width_subpixel(img, c)
print(f'ความกว้างจริง         {PART_W * PPM_TRUE:.2f} px  = {PART_W:.2f} มม.')
print(f'minAreaRect (เต็มพิกเซล) {w_rect:.2f} px  = {w_rect / PPM_TRUE:.2f} มม.   ผิด {abs(w_rect / PPM_TRUE - PART_W):.3f} มม.')
print(f'Sub-pixel (50% crossing)  {w_sub:.2f} px  = {w_sub / PPM_TRUE:.2f} มม.   ผิด {abs(w_sub / PPM_TRUE - PART_W):.3f} มม.')

**สังเกต:** วิธี Sub-pixel ผิดระดับ 0.03 มม. (ราว 0.3 พิกเซล) และ**ไม่แย่ลงเมื่อภาพเบลอ** ส่วน Threshold จะผิดมากขึ้นตามความเบลอ (ลอง `make_scene(blur=4)`)  เงื่อนไข: ขอบต้องเป็น**เส้นตรงหรือรู้รูปทรง** แสงสม่ำเสมอ และสแกนหลายเส้นแล้วเฉลี่ย
OpenCV มีตัวช่วย: `cv2.cornerSubPix` สำหรับมุม และ `cv2.fitLine` / `cv2.fitEllipse` สำหรับขอบตรงและวงกลม ซึ่งเฉลี่ยจากหลายจุดจึงละเอียดกว่าพิกเซลเช่นกัน

## 5. ทดสอบความทำซ้ำได้ (Repeatability)
ถ่ายชิ้นเดิม 30 ครั้ง (จุดรบกวนต่างกัน) แล้วดูว่าค่าวัดกระจายแค่ไหน  **ค่าที่รายงานได้ = เฉลี่ย ± 3σ**  ถ้า 3σ ใหญ่กว่าพิกัดความเผื่อ (Tolerance) ของชิ้นงาน ระบบนี้ใช้คัดไม่ได้

In [ ]:
TOL = 0.10   # มม. พิกัดความเผื่อของชิ้นงาน (สมมติ ±0.10)
res_rect, res_sub = [], []
for _ in range(30):
    im = make_scene()
    c = contour_of_part(im)
    res_rect.append(max(cv2.minAreaRect(c)[1]) / PPM_TRUE)
    res_sub.append(width_subpixel(im, c) / PPM_TRUE)
for name, r in [('minAreaRect', res_rect), ('Sub-pixel', res_sub)]:
    r = np.array(r)
    bias = r.mean() - PART_W
    ok = 'ใช้คัดได้' if 3 * r.std() < TOL else 'ใช้คัดไม่ได้ (3σ > Tolerance)'
    print(f'{name:12s} เฉลี่ย {r.mean():.3f} มม.  bias {bias:+.3f}  σ {r.std():.4f}  3σ {3 * r.std():.3f} มม.  -> {ok}')
print(f'(Tolerance ±{TOL} มม.  ขนาดพิกเซล {1 / PPM_TRUE:.3f} มม.)')

**อ่านผล:** `bias` คือความคลาดเคลื่อนเป็นระบบ (แก้ได้โดยสอบเทียบชดเชย)  `3σ` คือความกระจายที่แก้ไม่ได้ ต้องลดด้วยแสง โฟกัส และวิธีวัดที่ดีขึ้น  ในงานจริงใช้เกณฑ์ **Gauge R&R: 3σ ต้องน้อยกว่า 10–30% ของช่วง Tolerance**

## 6. เช็กลิสต์ก่อนวัดจริง และขีดจำกัด
**ติดตั้ง**
1. กล้องตั้งฉากกับระนาบชิ้นงาน ใช้ระดับน้ำ  หรือติดจุดอ้างอิง 4 จุดแล้วแก้ด้วย Homography (ขั้น 3)
2. **Backlight** (ไฟจากด้านหลัง) ให้ได้เงาดำขอบคมที่สุด  ถ้าทำไม่ได้ใช้ไฟโดมกระจายแสงเท่ากันทั้งภาพ
3. โฟกัสที่ขอบชิ้นงาน ไม่ใช่ที่พื้น  รูรับแสง f/4–f/8 เพื่อให้ช่วงชัดลึกพอ
4. วัตถุสอบเทียบ**ระนาบเดียวกับชิ้นงาน** และมีขนาดใกล้เคียงกัน (เหรียญใช้ทดลอง งานจริงใช้ Gauge block หรือ Calibration target)

**ก่อนใช้งาน**
5. สอบเทียบด้วยวัตถุที่รู้ขนาดอย่างน้อย 2 ขนาด ที่ 3 ตำแหน่งในภาพ (กลาง ซ้าย ขวา) ถ้าค่าต่างกัน > 0.5% ต้องแก้ Distortion ด้วย `cv2.calibrateCamera`
6. ทำ Repeatability 30 ครั้ง (ขั้น 5) และเทียบกับเวอร์เนียร์หรือไมโครมิเตอร์
7. สอบเทียบซ้ำทุกกะ หรือทุกครั้งที่ขยับกล้อง/เปลี่ยนเลนส์

**ขีดจำกัดที่ต้องรู้**
| ต้องการ | ทำได้ด้วย |
|---|---|
| ±0.5 มม. ชิ้นงานแบน | กล้อง 5 MP + เลนส์ธรรมดา + จุดอ้างอิง 4 จุด (ชุดนี้) |
| ±0.05 มม. | Backlight + Sub-pixel + ซูมให้ชิ้นงานเต็มภาพ + สอบเทียบ Distortion |
| ±0.01 มม. หรือวัตถุสูง | **เลนส์ Telecentric** (ขนาดในภาพไม่เปลี่ยนตามระยะ) + Backlight แบบ Telecentric |
| ความสูง ปริมาตร | กล้อง 3 มิติ / Laser profiler |

**สูตรเริ่มต้น:** ขนาดพิกเซล (มม.) = FOV (มม.) ÷ จำนวนพิกเซล  ต้องการความแม่น X มม. → ขนาดพิกเซลควรเล็กกว่า X/3 (Threshold) หรือ X (Sub-pixel)

<details>
<summary><b>โค้ดสอบเทียบ Distortion ด้วยลายหมากรุก (ไม่ได้รันในนี้)</b></summary>

```python
objp = np.zeros((9 * 6, 3), np.float32)
objp[:, :2] = np.mgrid[0:9, 0:6].T.reshape(-1, 2) * 10.0        # ช่องละ 10 มม.
obj_pts, img_pts = [], []
for path in checkerboard_images:                                    # ถ่าย 10-15 ภาพ หลายมุม
    gray = cv2.imread(path, 0)
    ok, corners = cv2.findChessboardCorners(gray, (9, 6))
    if ok:
        corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_COUNT, 30, 0.001))
        obj_pts.append(objp); img_pts.append(corners)
ret, K, dist, _, _ = cv2.calibrateCamera(obj_pts, img_pts, gray.shape[::-1], None, None)
undistorted = cv2.undistort(img, K, dist)                          # ใช้ก่อนวัดทุกภาพ
```
</details>